# 02. Xây dựng và Kiểm chứng Pipeline Dữ liệu DurianLDD (Data Pipeline)

Notebook này thiết lập, tối ưu hóa và kiểm chứng toàn diện pipeline nạp dữ liệu PyTorch cho bộ dữ liệu lá sầu riêng DurianLDD (4.437 ảnh). Quy trình bao gồm kiểm tra tính toàn vẹn của tệp trên đĩa, cấu hình Transforms tiền xử lý (chỉ tăng cường hình học ở Train, không tăng cường ở Val/Test), nạp dữ liệu qua Dataset/DataLoader chuẩn, phân tích thống kê kênh và trực quan hóa mẫu ảnh.

---

## 0. Cấu hình thực nghiệm (Config)

**Mục đích:** Xác định `PROJECT_ROOT` (thư mục chứa `README.md`) linh hoạt không hardcode đường dẫn, đọc toàn bộ tham số cấu hình từ `configs/pipeline.yaml`, xác nhận `image_root` trỏ đúng thư mục chứa `DLD_FinalDataset_224_spit` và in ra 5 đường dẫn đầu tiên ghép với `image_root` để kiểm tra trực quan.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import yaml

pd.set_option("mode.string_storage", "python")

def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current] + list(current.parents):
        if (candidate / "README.md").is_file():
            return candidate
    raise FileNotFoundError("Không tìm thấy thư mục gốc dự án chứa README.md!")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / "configs" / "pipeline.yaml"
assert CONFIG_PATH.is_file(), f"Không tìm thấy tệp cấu hình: {CONFIG_PATH}"

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

required_keys = [
    "seed",
    "image_root",
    "image_size",
    "batch_size",
    "num_workers",
    "pin_memory",
    "mean",
    "std",
    "hflip_prob",
    "vflip_prob",
    "rotation_degrees",
    "num_classes",
    "expected_train_count",
    "expected_val_count",
    "classes",
]
for key in required_keys:
    assert key in config, f"Thiếu khóa cấu hình bắt buộc: '{key}' trong {CONFIG_PATH}"

SEED = config["seed"]
IMAGE_ROOT = PROJECT_ROOT / config["image_root"]
IMAGE_SIZE = config["image_size"]
BATCH_SIZE = config["batch_size"]
NUM_WORKERS = config["num_workers"]
PIN_MEMORY = config["pin_memory"]
MEAN = config["mean"]
STD = config["std"]
NUM_CLASSES = config["num_classes"]
EXPECTED_TRAIN_COUNT = config["expected_train_count"]
EXPECTED_VAL_COUNT = config["expected_val_count"]
CLASSES = config["classes"]

assert len(CLASSES) == NUM_CLASSES, f"Số lượng lớp ({len(CLASSES)}) không khớp với num_classes ({NUM_CLASSES})"
assert len(MEAN) == 3 and len(STD) == 3, "Thông số mean và std phải có đúng 3 kênh RGB"

assert (IMAGE_ROOT / "DLD_FinalDataset_224_spit").is_dir(), f"Không tìm thấy DLD_FinalDataset_224_spit tại {IMAGE_ROOT}"

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"Config file  : {CONFIG_PATH.name}")
print(f"Image root   : {IMAGE_ROOT.relative_to(PROJECT_ROOT)}")
print(f"Random seed  : {SEED}")
print(f"Image size   : {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"Batch size   : {BATCH_SIZE}")
print(f"Classes ({NUM_CLASSES}): {CLASSES}")

manifest_df = pd.read_csv(PROJECT_ROOT / "manifests" / "split_manifest.csv")
print("\n=== 5 ĐƯỜNG DẪN ĐẦU TIÊN CỦA MANIFEST GHÉP VỚI IMAGE_ROOT ===")
for i, p in enumerate(manifest_df["image_path"].head(5)):
    full_p = IMAGE_ROOT / p
    print(f"[{i}] {full_p} (Tồn tại: {full_p.is_file()})")

PROJECT_ROOT : C:\WorkSpace\Study\durian-leaf-disease-robustness
Config file  : pipeline.yaml
Image root   : datasets\durian-ldd
Random seed  : 42
Image size   : 224x224
Batch size   : 32
Classes (5): ['ALGAL_LEAF_SPOT', 'ALLOCARIDARA_ATTACK', 'HEALTHY_LEAF', 'LEAF_BLIGHT', 'PHOMOPSIS_LEAF_SPOT']

=== 5 ĐƯỜNG DẪN ĐẦU TIÊN CỦA MANIFEST GHÉP VỚI IMAGE_ROOT ===
[0] C:\WorkSpace\Study\durian-leaf-disease-robustness\datasets\durian-ldd\DLD_FinalDataset_224_spit\test\ALGAL_LEAF_SPOT\to_label_100.jpg (Tồn tại: True)
[1] C:\WorkSpace\Study\durian-leaf-disease-robustness\datasets\durian-ldd\DLD_FinalDataset_224_spit\test\ALGAL_LEAF_SPOT\to_label_104.jpg (Tồn tại: True)
[2] C:\WorkSpace\Study\durian-leaf-disease-robustness\datasets\durian-ldd\DLD_FinalDataset_224_spit\test\ALGAL_LEAF_SPOT\to_label_108.jpg (Tồn tại: True)
[3] C:\WorkSpace\Study\durian-leaf-disease-robustness\datasets\durian-ldd\DLD_FinalDataset_224_spit\test\ALGAL_LEAF_SPOT\to_label_12.jpg (Tồn tại: True)
[4] C:\WorkSpace\Study\d

## 1. Cấu hình Transforms (Train và Eval)

**Mục đích:** Khởi tạo pipeline transform từ `configs/pipeline.yaml`. Kiểm tra cấu trúc gồm 6 bước cho `train_transform` (chỉ tăng cường hình học: lật ngang, lật dọc, xoay góc) và 3 bước cho `eval_transform` (chỉ resize khi cần, ToTensor, Normalize; dùng chung cho val và test).

In [2]:
from src.data.transforms import (
    ConditionalResize,
    build_train_transform,
    build_eval_transform,
)

train_transform = build_train_transform(config)
eval_transform = build_eval_transform(config)

assert len(train_transform.transforms) == 6, f"Train transform phải có đúng 6 bước, hiện có {len(train_transform.transforms)}"
assert len(eval_transform.transforms) == 3, f"Eval transform phải có đúng 3 bước, hiện có {len(eval_transform.transforms)}"

assert isinstance(train_transform.transforms[0], ConditionalResize), "Bước 1 train transform phải là ConditionalResize"
assert isinstance(eval_transform.transforms[0], ConditionalResize), "Bước 1 eval transform phải là ConditionalResize"

print("=== TRAIN TRANSFORM ===")
print(train_transform)
print("\n=== EVAL TRANSFORM (VAL & TEST) ===")
print(eval_transform)

=== TRAIN TRANSFORM ===
Compose(
    ConditionalResize(size=(224, 224))
    RandomHorizontalFlip(p=0.5)
    RandomVerticalFlip(p=0.5)
    RandomRotation(degrees=[-15.0, 15.0], interpolation=nearest, expand=False, fill=0)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)

=== EVAL TRANSFORM (VAL & TEST) ===
Compose(
    ConditionalResize(size=(224, 224))
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)


## 2. Kiểm tra đường dẫn toàn bộ 4.437 ảnh (Path Verification)

**Mục đích:** Duyệt toàn bộ 4.437 đường dẫn tệp trong manifest, chỉ kiểm tra sự tồn tại của tệp trên đĩa bằng `Path.is_file()`. Tuyệt đối KHÔNG mở ảnh test và KHÔNG gọi `get_split("test")`. Dừng ngay nếu phát hiện đường dẫn bị thiếu.

In [3]:
manifest_df = pd.read_csv(PROJECT_ROOT / "manifests" / "split_manifest.csv")
assert len(manifest_df) == 4437, f"Tổng số ảnh trong manifest ({len(manifest_df)}) != 4437 kỳ vọng!"

missing_paths = [
    p for p in manifest_df["image_path"]
    if not (IMAGE_ROOT / p).is_file()
]

if missing_paths:
    print(f"LỖI: Phát hiện {len(missing_paths)} tệp ảnh bị thiếu trên đĩa! 10 đường dẫn đầu tiên:")
    for mp in missing_paths[:10]:
        print("  -", mp)
    assert len(missing_paths) == 0, f"Có {len(missing_paths)} tệp ảnh bị thiếu trên đĩa!"

print(f"Tổng số ảnh kiểm tra trong manifest : {len(manifest_df)}")
print(f"Số ảnh tồn tại hợp lệ trên đĩa       : {len(manifest_df) - len(missing_paths)}/4437 (100%)")
print("Xác nhận: Tất cả 4.437 tệp ảnh trong manifest đều tồn tại đầy đủ trên đĩa!")

Tổng số ảnh kiểm tra trong manifest : 4437
Số ảnh tồn tại hợp lệ trên đĩa       : 4437/4437 (100%)
Xác nhận: Tất cả 4.437 tệp ảnh trong manifest đều tồn tại đầy đủ trên đĩa!


## 3. Quét kích thước và định dạng ảnh Train & Val

**Mục đích:** Quét toàn bộ 3.549 ảnh thuộc tập train (3.105) và tập val (444) thông qua `get_split()` (không đọc ảnh test). Kiểm tra kích thước ảnh gốc có đúng chuẩn 224x224 hay không, kiểm tra hệ màu RGB, xác nhận không có ảnh hỏng và xác nhận `ConditionalResize` không can thiệp nội suy khi kích thước ảnh đã chuẩn.

In [4]:
from PIL import Image
from src.data.splits import get_split

train_df = get_split("train")
val_df = get_split("val")
scanned_df = pd.concat([train_df, val_df], ignore_index=True)

non_224_count = 0
non_rgb_count = 0
unreadable_count = 0
resize_skipped_count = 0
resizer = train_transform.transforms[0]

for rel_path in scanned_df["image_path"]:
    img_path = IMAGE_ROOT / rel_path
    try:
        with Image.open(img_path) as img:
            if img.size != (224, 224):
                non_224_count += 1
            if img.mode != "RGB":
                non_rgb_count += 1
            if resizer(img) is img:
                resize_skipped_count += 1
    except Exception:
        unreadable_count += 1

print(f"Tổng số ảnh train + val đã quét  : {len(scanned_df)}")
print(f"Số ảnh không phải 224x224          : {non_224_count}")
print(f"Số ảnh không phải hệ màu RGB      : {non_rgb_count}")
print(f"Số ảnh không mở được              : {unreadable_count}")
print(f"Số ảnh Resize bỏ qua (giữ nguyên) : {resize_skipped_count}/{len(scanned_df)}")

assert non_224_count == 0, f"Có {non_224_count} ảnh không đúng 224x224!"
assert non_rgb_count == 0, f"Có {non_rgb_count} ảnh không phải RGB!"
assert unreadable_count == 0, f"Có {unreadable_count} ảnh bị lỗi!"
assert resize_skipped_count == len(scanned_df), "ConditionalResize phải bỏ qua khi ảnh đã đúng 224x224!"

Tổng số ảnh train + val đã quét  : 3549
Số ảnh không phải 224x224          : 0
Số ảnh không phải hệ màu RGB      : 0
Số ảnh không mở được              : 0
Số ảnh Resize bỏ qua (giữ nguyên) : 3549/3549


## 4. Dataset, DataLoader và nạp batch đầu tiên

**Mục đích:** Khởi tạo `train_loader` và `val_loader` thông qua `build_dataloaders(config, include_test=False)`. Đối soát số lượng ảnh và phân bố lớp; nạp batch đầu tiên từ `train_loader` để xác thực shape `(32, 3, 224, 224)`, kiểu dữ liệu `torch.float32`, dải giá trị min/max và thống kê mean/std theo kênh RGB sau chuẩn hóa ImageNet.

In [5]:
import torch
from src.data.dataloaders import build_dataloaders

loaders = build_dataloaders(config, include_test=False)
train_loader = loaders["train"]
val_loader = loaders["val"]

train_ds = train_loader.dataset
val_ds = val_loader.dataset

assert len(train_ds) == EXPECTED_TRAIN_COUNT, f"Số ảnh train ({len(train_ds)}) != {EXPECTED_TRAIN_COUNT}"
assert len(val_ds) == EXPECTED_VAL_COUNT, f"Số ảnh val ({len(val_ds)}) != {EXPECTED_VAL_COUNT}"

print(f"Train Dataset: {len(train_ds)} ảnh | {len(train_loader)} batches (batch_size={BATCH_SIZE})")
print(f"Val Dataset  : {len(val_ds)} ảnh | {len(val_loader)} batches (batch_size={BATCH_SIZE})")

print("\n=== PHÂN BỐ LỚP TRONG TẬP TRAIN ===")
print(train_ds.split_df["class_name"].value_counts())

print("\n=== PHÂN BỐ LỚP TRONG TẬP VAL ===")
print(val_ds.split_df["class_name"].value_counts())

first_images, first_labels, first_paths = next(iter(train_loader))
print("\n=== THÔNG TIN BATCH ĐẦU TIÊN (TRAIN) ===")
print(f"Images shape : {first_images.shape}")
print(f"Images dtype : {first_images.dtype}")
print(f"Images min   : {first_images.min().item():.4f}")
print(f"Images max   : {first_images.max().item():.4f}")
print(f"Labels shape : {first_labels.shape}")
print(f"Sample paths : {first_paths[:2]}")

assert first_images.shape == (BATCH_SIZE, 3, 224, 224), f"Shape batch không đúng: {first_images.shape}"
assert first_images.dtype == torch.float32, f"Dtype không đúng: {first_images.dtype}"

means, stds = [], []
for i, (imgs, _, _) in enumerate(train_loader):
    if i >= 5:
        break
    means.append(imgs.mean(dim=[0, 2, 3]))
    stds.append(imgs.std(dim=[0, 2, 3]))
channel_means = torch.stack(means).mean(dim=0)
channel_stds = torch.stack(stds).mean(dim=0)

print("\n=== MEAN & STD THEO KÊNH (RGB) SAU NORMALIZE (5 BATCHES) ===")
for ch, m, s in zip(["R", "G", "B"], channel_means, channel_stds):
    print(f"Kênh {ch}: Mean = {m.item():+.4f} (kỳ vọng ~0), Std = {s.item():.4f} (kỳ vọng ~1)")
    assert abs(m.item()) < 1.0, f"Mean kênh {ch} sai lệch quá lớn so với 0: {m.item()}"
    assert 0.5 < s.item() < 1.5, f"Std kênh {ch} sai lệch quá lớn so với 1: {s.item()}"

Train Dataset: 3105 ảnh | 98 batches (batch_size=32)
Val Dataset  : 444 ảnh | 14 batches (batch_size=32)

=== PHÂN BỐ LỚP TRONG TẬP TRAIN ===
class_name
HEALTHY_LEAF           683
LEAF_BLIGHT            657
ALLOCARIDARA_ATTACK    638
PHOMOPSIS_LEAF_SPOT    614
ALGAL_LEAF_SPOT        513
Name: count, dtype: int64

=== PHÂN BỐ LỚP TRONG TẬP VAL ===
class_name
HEALTHY_LEAF           97
LEAF_BLIGHT            93
ALLOCARIDARA_ATTACK    92
PHOMOPSIS_LEAF_SPOT    88
ALGAL_LEAF_SPOT        74
Name: count, dtype: int64

=== THÔNG TIN BATCH ĐẦU TIÊN (TRAIN) ===
Images shape : torch.Size([32, 3, 224, 224])
Images dtype : torch.float32
Images min   : -2.1179
Images max   : 2.6400
Labels shape : torch.Size([32])
Sample paths : ['DLD_FinalDataset_224_spit/train/ALGAL_LEAF_SPOT/to_label_33.jpg', 'DLD_FinalDataset_224_spit/train/HEALTHY_LEAF/to_label_1961.jpg']



=== MEAN & STD THEO KÊNH (RGB) SAU NORMALIZE (5 BATCHES) ===
Kênh R: Mean = -0.5869 (kỳ vọng ~0), Std = 1.0112 (kỳ vọng ~1)
Kênh G: Mean = +0.0968 (kỳ vọng ~0), Std = 0.9545 (kỳ vọng ~1)
Kênh B: Mean = -0.2057 (kỳ vọng ~0), Std = 1.0396 (kỳ vọng ~1)


## 5. Trực quan hóa mẫu ảnh và lưu biểu đồ

**Mục đích:** Xây dựng lưới ảnh so sánh trực quan: 8 ảnh train gốc vs sau khi áp dụng `train_transform` (đã de-normalize về không gian RGB [0, 1] để quan sát tác động của lật ngang/dọc/xoay góc) và 8 ảnh validation. Lưu biểu đồ vào `outputs/figures/pipeline_samples.png`.

In [6]:
import matplotlib.pyplot as plt
import numpy as np

FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
SAMPLE_FIG_PATH = FIGURES_DIR / "pipeline_samples.png"

def denormalize(tensor: torch.Tensor, mean=MEAN, std=STD) -> np.ndarray:
    t = tensor.clone().detach().cpu()
    for c in range(3):
        t[c] = t[c] * std[c] + mean[c]
    t = torch.clamp(t, 0.0, 1.0)
    return t.permute(1, 2, 0).numpy()

num_samples = 8
fig, axes = plt.subplots(3, num_samples, figsize=(18, 7))

for idx in range(num_samples):
    row = train_ds.split_df.iloc[idx]
    with Image.open(IMAGE_ROOT / row["image_path"]) as raw_img:
        raw_rgb = raw_img.convert("RGB")
    axes[0, idx].imshow(raw_rgb)
    axes[0, idx].set_title(f"Train Raw\n{row['class_name'][:12]}", fontsize=9)
    axes[0, idx].axis("off")

    aug_tensor, _, _ = train_ds[idx]
    axes[1, idx].imshow(denormalize(aug_tensor))
    axes[1, idx].set_title("Train Aug", fontsize=9)
    axes[1, idx].axis("off")

    val_tensor, val_label, _ = val_ds[idx]
    val_row = val_ds.split_df.iloc[idx]
    axes[2, idx].imshow(denormalize(val_tensor))
    axes[2, idx].set_title(f"Val\n{val_row['class_name'][:12]}", fontsize=9)
    axes[2, idx].axis("off")

plt.tight_layout()
plt.savefig(SAMPLE_FIG_PATH, dpi=150, bbox_inches="tight")
plt.close()

assert SAMPLE_FIG_PATH.is_file(), f"Không tìm thấy tệp ảnh biểu đồ tại: {SAMPLE_FIG_PATH}"
print(f"Đã xuất và lưu thành công lưới ảnh mẫu tại: {SAMPLE_FIG_PATH.relative_to(PROJECT_ROOT)}")

Đã xuất và lưu thành công lưới ảnh mẫu tại: outputs\figures\pipeline_samples.png


## 6. Kiểm tra tính tái lập và các ràng buộc cốt lõi

**Mục đích:** Xác thực tính xác định tuyệt đối (deterministic) của `val_loader`, tính ngẫu nhiên có kiểm soát của `train_loader` khi đồng bộ seed, đảm bảo không có thao tác xử lý ảnh cấm trong pipeline chuẩn và xác nhận tập `test` bị khóa an toàn bởi `FINAL_EVAL=1`.

In [7]:
val_loader_run1 = build_dataloaders(config, include_test=False)["val"]
val_loader_run2 = build_dataloaders(config, include_test=False)["val"]
val_batch1 = next(iter(val_loader_run1))[0]
val_batch2 = next(iter(val_loader_run2))[0]
assert torch.equal(val_batch1, val_batch2), "Val loader phải cho kết quả giống hệt nhau giữa các lần chạy!"
print("[PASS] 1. Val loader có tính xác định tuyệt đối (deterministic) giữa hai lần chạy.")

torch.manual_seed(SEED)
train_loader_run1 = build_dataloaders(config, include_test=False)["train"]
train_batch1 = next(iter(train_loader_run1))

torch.manual_seed(SEED)
train_loader_run2 = build_dataloaders(config, include_test=False)["train"]
train_batch2 = next(iter(train_loader_run2))

assert train_batch1[2] == train_batch2[2], "Thứ tự đường dẫn ảnh trong batch train phải khớp hoàn toàn!"
assert torch.equal(train_batch1[1], train_batch2[1]), "Nhãn lớp trong batch train phải khớp hoàn toàn!"
assert torch.equal(train_batch1[0], train_batch2[0]), "Tensor ảnh train sau augmentation phải khớp hoàn toàn khi cùng seed!"
print("[PASS] 2. Train loader có ngẫu nhiên nhưng lặp lại 100% khi cùng random seed.")

forbidden_ops = {"colorjitter", "gamma", "clahe", "retinex", "denoise", "gaussianblur", "medianblur", "sharpen"}
for tf in list(train_transform.transforms) + list(eval_transform.transforms):
    cls_name = tf.__class__.__name__.lower()
    for f_op in forbidden_ops:
        assert f_op not in cls_name, f"Phát hiện thao tác cấm trong transform: {tf}"
print("[PASS] 3. Pipeline chuẩn tuyệt đối không chứa thao tác cấm (chỉ biến đổi hình học thuần túy).")

try:
    get_split("test")
    raise AssertionError("get_split('test') phải bị chặn khi chưa có FINAL_EVAL=1!")
except PermissionError:
    pass
print("[PASS] 4. Khóa an toàn hoạt động chính xác: get_split('test') bị chặn với PermissionError.")

print("\n===> TẤT CẢ CÁC MỤC KIỂM TRA PIPELINE TRÊN ẢNH THẬT ĐÃ HOÀN TẤT VÀ ĐẠT 100%!")

[PASS] 1. Val loader có tính xác định tuyệt đối (deterministic) giữa hai lần chạy.


[PASS] 2. Train loader có ngẫu nhiên nhưng lặp lại 100% khi cùng random seed.
[PASS] 3. Pipeline chuẩn tuyệt đối không chứa thao tác cấm (chỉ biến đổi hình học thuần túy).
[PASS] 4. Khóa an toàn hoạt động chính xác: get_split('test') bị chặn với PermissionError.

===> TẤT CẢ CÁC MỤC KIỂM TRA PIPELINE TRÊN ẢNH THẬT ĐÃ HOÀN TẤT VÀ ĐẠT 100%!
